# Studio sistematico comune (frequentista) e veti sul charm

[Giacomo, 2026-10-06] Notebook separato da `Frequentist_Analysis_Bis.ipynb`, per non toccarlo. Stessa lista di varianti e stessa regola dell'analisi bayesiana (`bayes:.../bayesian_nominal.py`):

- **varianti** (ognuna = modello nominale con un solo cambiamento): finestra con estremo basso 5050 e 5150, alto 5450 e 5700; un solo esponenziale; forma del fondo condivisa tra le cariche; rese di fondo simmetriche; segnale Crystal Ball con code libere; $\mu,\sigma$ separati per carica; preselezione ProbK $>0.4$ e $>0.6$, ProbPi $<0.4$ e $<0.6$ (uno alla volta);
- **regola**: per le varianti che cambiano il campione (finestra, preselezione) lo spostamento conta solo se supera $\sigma_B=\sqrt{|\sigma_{var}^2-\sigma_{nom}^2|}$ (Barlow); varianti sullo stesso campione con $\chi^2$ peggiore del nominale di più di 25 sono escluse (respinte dai dati); massimo $|\Delta|$ per gruppo, gruppi in quadratura (`lf.combine_systematics`);
- **veti sul charm** (controllo informativo, fuori dalla sistematica): solo $D^0$, e $D^0+J/\psi+\chi_{c0}$ (riferimento), masse PDG, $\pm30$ MeV per $D^0$ e $J/\psi$, $\pm60$ MeV per $\chi_{c0}$, sulle due coppie $K^+K^-$ di carica opposta.

Percorso dei dati: variabile d'ambiente `LHCB_DATA_DIR` (default il percorso di Marco).

In [1]:
import os, json, warnings
from pathlib import Path
from dataclasses import replace
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=RuntimeWarning)
import lhcb_freq as lf

_DATA = Path(os.environ.get("LHCB_DATA_DIR", r"C:\\Users\\marco\\Documents\\Università\\Anno IV\\Statistics_and_Data_Analysis\\Data"))
cfg = lf.Config(data_dir=_DATA, sim_file=_DATA / "PhaseSpaceSimulation.root")
RESULTS_DIR = os.path.join(os.getcwd(), "results"); os.makedirs(RESULTS_DIR, exist_ok=True)

data = lf.load_data(cfg)[cfg.columns]
# campione con tagli piu' larghi di tutte le varianti: scan_config riapplica la preselezione di ogni variante
data_loose = data[lf.kaon_preselection(data, replace(cfg, prob_k_min=0.4, prob_pi_max=0.6))].reset_index(drop=True)
lf.add_B_variables(data_loose, cfg)
nominal_sample = lf.select_window(data_loose[lf.kaon_preselection(data_loose, cfg)], cfg.fit_window)
print("candidati nella finestra nominale:", len(nominal_sample))

/opt/homebrew/Caskroom/miniforge/base/envs/cern/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


candidati nella finestra nominale: 19731


## Varianti

In [2]:
VARIATIONS = [dict(label="nominal"),
    dict(label="window_lo5050", fit_window=(5050.0, 5600.0)), dict(label="window_lo5150", fit_window=(5150.0, 5600.0)),
    dict(label="window_hi5450", fit_window=(5100.0, 5450.0)), dict(label="window_hi5700", fit_window=(5100.0, 5700.0)),
    dict(label="bkg_single_exp", n_exp=1), dict(label="bkg_shared_shape", bkg_shape="shared"),
    dict(label="bkg_symmetric", bkg_yield="symmetric"),
    dict(label="signal_crystal_ball", signal="cb"), dict(label="signal_per_charge", signal_per_charge=True),
    dict(label="presel_probk_0.4", prob_k_min=0.4), dict(label="presel_probk_0.6", prob_k_min=0.6),
    dict(label="presel_probpi_0.4", prob_pi_max=0.4), dict(label="presel_probpi_0.6", prob_pi_max=0.6)]
GROUPS = {"window": ["window_lo5050", "window_lo5150", "window_hi5450", "window_hi5700"],
          "background": ["bkg_single_exp", "bkg_shared_shape", "bkg_symmetric"],
          "signal": ["signal_crystal_ball", "signal_per_charge"],
          "preselection": ["presel_probk_0.4", "presel_probk_0.6", "presel_probpi_0.4", "presel_probpi_0.6"]}
SAMPLE_CHANGING = GROUPS["window"] + GROUPS["preselection"]

res = lf.scan_config(data_loose, cfg, VARIATIONS)
cols = ["label", "valid", "accurate", "n_events", "A_CP", "A_CP_err", "chi2"]
print(res[cols].to_string())
nom = res[res.label == "nominal"].iloc[0]
syst = lf.combine_systematics(nom, [r for _, r in res.iterrows() if r.label != "nominal"], GROUPS, SAMPLE_CHANGING)
for k, v in syst["variations"].items():
    print(f"{k:22s} shift={v['shift']:+.4f}  sigma_B={v['sigma_barlow'] if v['sigma_barlow'] is None else round(v['sigma_barlow'], 4)}"
          f"  {'ESCLUSA (dati)' if v['rejected_by_data'] else ('conta' if v['counts'] else 'non conta (Barlow)')}")
print("per gruppo:", {k: round(v, 4) for k, v in syst["groups"].items()}, " totale:", round(syst["total"], 4),
      " (statistica:", round(syst["sigma_stat"], 4), ")")

scan:   0%|          | 0/14 [00:00<?, ?it/s]

scan:   7%|▋         | 1/14 [00:01<00:25,  1.93s/it]

scan:  14%|█▍        | 2/14 [00:03<00:23,  1.98s/it]

scan:  21%|██▏       | 3/14 [00:05<00:18,  1.64s/it]

scan:  29%|██▊       | 4/14 [00:06<00:16,  1.62s/it]

...

scan:  86%|████████▌ | 12/14 [00:18<00:03,  1.51s/it]

scan:  93%|█████████▎| 13/14 [00:19<00:01,  1.44s/it]

scan: 100%|██████████| 14/14 [00:21<00:00,  1.46s/it]

scan: 100%|██████████| 14/14 [00:21<00:00,  1.51s/it]

                  label  valid  accurate  n_events      A_CP  A_CP_err        chi2
0               nominal   True      True     19731 -0.044129  0.008499  441.716011
1         window_lo5050   True      True     21164 -0.044503  0.008512  519.255452
2         window_lo5150   True     False     18566 -0.044590  0.004306  455.377177
3         window_hi5450   True      True     19004 -0.044222  0.008492  482.733917
4         window_hi5700   True      True     20191 -0.045537  0.008500  473.802492
5        bkg_single_exp   True      True     19731 -0.048060  0.008471  576.264524
6      bkg_shared_shape   True      True     19731 -0.048143  0.008342  451.063731
7         bkg_symmetric   True      True     19731 -0.047238  0.008387  448.433676
8   signal_crystal_ball   True      True     19731 -0.044729  0.008477  406.794596
9     signal_per_charge   True      True     19731 -0.044245  0.008668  441.464469
10     presel_probk_0.4   True      True     26553 -0.047306  0.007640  444.521015
11  

## Veti sul charm (controllo informativo)

In [3]:
vetoes = {}
for name, mode in (("veto_D0", "D"), ("veto_D0_Jpsi_chic0", "all")):
    d = lf.remove_charm_resonances(cfg, nominal_sample, mode)
    m = lf.fit_joint(lf.split_by_charge(d, col="B_Mass"), cfg)
    vetoes[name] = dict(n=int(len(d)), A_CP=float(m.values["A_CP"]), A_CP_err=float(m.errors["A_CP"]),
                        shift=float(m.values["A_CP"] - nom.A_CP), valid=bool(m.valid), accurate=bool(m.accurate))
    print(f"{name:20s} n={len(d)}  A_raw = {m.values['A_CP']:.4f} +- {m.errors['A_CP']:.4f}  (spostamento {m.values['A_CP'] - nom.A_CP:+.4f})")
assert vetoes["veto_D0"]["n"] == 18744 and vetoes["veto_D0_Jpsi_chic0"]["n"] == 16367, "conteggi diversi da quelli concordati"

veto_D0              n=18744  A_raw = -0.0493 +- 0.0087  (spostamento -0.0052)


veto_D0_Jpsi_chic0   n=16367  A_raw = -0.0421 +- 0.0094  (spostamento +0.0021)


## Salvataggio

In [4]:
def _clean(x):
    if isinstance(x, (np.floating, np.integer, np.bool_)):
        return x.item()
    return x
out = {"nominal": {k: _clean(nom[k]) for k in ("A_CP", "A_CP_err", "chi2", "n_events")},
       "variations": [{k: _clean(v) for k, v in r.items() if k != "gof"} for _, r in res.iterrows()],
       "systematics": syst, "charm_vetoes": vetoes, "groups": GROUPS, "sample_changing": SAMPLE_CHANGING}
with open(os.path.join(RESULTS_DIR, "freq_systematics.json"), "w") as f:
    json.dump(out, f, indent=1, default=str)
print("[saved]", os.path.join(RESULTS_DIR, "freq_systematics.json"))

[saved] /Users/gmoretti/Work/wt_freq/LHCb/Code_general/Frequentist/results/freq_systematics.json
